# 03 - Ball values
Every ball gets a value = what happened minus what an average player does in that exact state (out-of-fold expectation).
Two components are kept separate because they have very different reliability:
  vr = runs above/below expectation,  vo = wicket event vs expectation, priced at the state-specific run cost of a wicket.
Outputs: outputs/batter_ball_values.parquet, outputs/bowler_ball_values.parquet

In [1]:
import warnings
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor as HGR, HistGradientBoostingClassifier as HGC
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_rows', 80)
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA, OUT = ROOT / 'data', ROOT / 'outputs'
b = pd.read_parquet(OUT / 'balls_pressure.parquet')

TIER_W = {'IPL': 1.0, 'T20I_FM': 1.0, 'T20I_ASSOC': 0.5, 'A_TEAM': 0.5}   # evidence weight by opposition quality

## 1. Expected outcome for an average player in the same state (out-of-fold, grouped by match)
Evaluation set: regulation innings of completed matches. No-result and DLS matches are excluded (state is unreliable).
Super overs are kept and valued against last-over expectations.

In [2]:
b['ps_c'] = b.pace_or_spin.map({'Pace': 0, 'Spin': 1})
b['bo_c'] = np.select([b.batting_order <= 2, b.batting_order <= 4, b.batting_order <= 6], [0, 1, 2], 3)
FE = ['innings', 'overid', 'wkts', 'balls_left', 'crr', 'rrr', 'runs_needed', 'bat_balls_before',
      'g_c', 'bt_c', 'ot_c', 'ps_c', 'bo_c']
CAT = [FE.index(c) for c in ['g_c', 'bt_c', 'ot_c', 'ps_c', 'bo_c']]

E = b[b.innings.isin([1, 2]) & b.status.isin(['ok', 'tied']) & ~b.dls & (b.balls_left >= 1)].copy()
perm = np.random.default_rng(7).permutation(E.match_id.unique())
E['fold'] = E.match_id.map({m: i % 5 for i, m in enumerate(perm)})


def oof(y, mask, kind):
    pred = pd.Series(np.nan, index=E.index)
    for f in range(5):
        tr, te = E[mask & (E.fold != f)], E[mask & (E.fold == f)]
        M = HGC if kind == 'c' else HGR
        mdl = M(max_iter=200, learning_rate=0.06, max_leaf_nodes=24, min_samples_leaf=200, l2_regularization=1.0,
                categorical_features=CAT, random_state=0).fit(tr[FE], y[tr.index])
        pred[te.index] = mdl.predict_proba(te[FE])[:, 1] if kind == 'c' else mdl.predict(te[FE])
    return pred


allrows = pd.Series(True, index=E.index)
E['exp_runs'] = oof(E.runs, E.faced, 'r')
E['exp_out'] = oof(E.batter_out.astype(int), E.faced, 'c')
E['exp_bruns'] = oof(E.bowler_runs, allrows, 'r')
E['exp_bwkt'] = oof(E.bowler_wkt.astype(int), allrows, 'c')

# super overs: expectation = regulation last-over averages
last = E[E.overid == 20]
SO = b[b.is_super_over].copy()
for c, src in [('exp_runs', last[last.faced].runs), ('exp_out', last[last.faced].batter_out), ('exp_bruns', last.bowler_runs), ('exp_bwkt', last.bowler_wkt)]:
    SO[c] = src.astype(float).mean()
SO['wkt_cost'] = last.wkt_cost.mean()
D = pd.concat([E, SO], ignore_index=True)
D['w'] = D.match_tier.map(TIER_W)
print('wicket cost (runs) by phase:', D.groupby('phase').wkt_cost.mean().round(1).to_dict())
print('OOF check, mean actual vs expected bat runs/ball: %.3f vs %.3f' % (E[E.faced].runs.mean(), E[E.faced].exp_runs.mean()))

wicket cost (runs) by phase: {'DEATH': 2.5, 'MID': 5.7, 'PP': 9.8, 'SO': 1.4}
OOF check, mean actual vs expected bat runs/ball: 1.393 vs 1.394


## 2. Ball values (runs-equivalent)

In [3]:
Bt = D[D.faced].copy()
Bt['vr'] = Bt.runs - Bt.exp_runs
Bt['vo'] = -(Bt.batter_out.astype(float) - Bt.exp_out) * Bt.wkt_cost
Bw = D.copy()
Bw['vr'] = -(Bw.bowler_runs - Bw.exp_bruns)
Bw['vo'] = (Bw.bowler_wkt.astype(float) - Bw.exp_bwkt) * Bw.wkt_cost
for X in (Bt, Bw):
    X['v'] = X.vr + X.vo

KEEP = ['match_id', 'innings', 'innings_ball_num', 'season', 'comp_group', 'match_tier', 'match_date', 'phase', 'overid',
        'striker_id', 'bowler_id', 'bat_team', 'bowl_team', 'runs', 'tot', 'legal', 'faced', 'batter_out', 'bowler_runs',
        'bowler_wkt', 'is_four', 'is_six', 'is_super_over', 'p_bat', 'p_bowl', 'w', 'v', 'vr', 'vo']
Bt[KEEP].to_parquet(OUT / 'batter_ball_values.parquet')
Bw[KEEP].to_parquet(OUT / 'bowler_ball_values.parquet')
print('batter ball SD: runs-part %.2f, wicket-part %.2f' % (Bt.vr.std(), Bt.vo.std()))
print('bowler ball SD: runs-part %.2f, wicket-part %.2f' % (Bw.vr.std(), Bw.vo.std()))
print('saved ball values:', len(Bt), 'batter rows,', len(Bw), 'bowler rows')

batter ball SD: runs-part 1.71, wicket-part 1.64
bowler ball SD: runs-part 1.69, wicket-part 1.60
saved ball values: 188097 batter rows, 195619 bowler rows
